In [122]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

## Описание данных

**Целевая переменная:** `SeriousDlqin2yrs` — был ли у клиента просрочка 90+ дней или хуже за последние 2 года.

| Признак | Описание | Тип данных |
|---------|----------|-------------|
| `SeriousDlqin2yrs` | Просрочка 90+ дней или хуже за последние 2 года | Y/N (целевой) |
| `RevolvingUtilizationOfUnsecuredLines` | Отношение общего остатка по кредитным картам и персональным кредитным линиям (кроме недвижимости и рассрочки) к сумме кредитных лимитов | доля |
| `age` | Возраст заёмщика в годах | целое число |
| `NumberOfTime30-59DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 30–59 дней (но не более) за последние 2 года | целое число |
| `DebtRatio` | Отношение ежемесячных выплат по долгам, алиментов, расходов на жизнь к валовому месячному доходу | доля |
| `MonthlyIncome` | Ежемесячный доход | вещественное число |
| `NumberOfOpenCreditLinesAndLoans` | Количество открытых кредитов (например, автокредит, ипотека) и кредитных линий (например, кредитные карты) | целое число |
| `NumberOfTimes90DaysLate` | Количество раз, когда заёмщик допускал просрочку 90+ дней | целое число |
| `NumberRealEstateLoansOrLines` | Количество ипотечных кредитов и кредитов под недвижимость, включая возобновляемые кредитные линии под залог недвижимости | целое число |
| `NumberOfTime60-89DaysPastDueNotWorse` | Количество раз, когда заёмщик допускал просрочку 60–89 дней (но не более) за последние 2 года | целое число |
| `NumberOfDependents` | Количество иждивенцев в семье (не включая самого заёмщика) — супруг(а), дети и т.д. | целое число |


In [123]:
train_file = 'data.csv'

In [124]:
df = pd.read_csv(train_file)
df.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents,SeriousDlqin2yrs
0,0.114987,62,0,1841.000000,NaN,5,0,1,0,2.0,0
1,0.008705,73,0,0.498553,3800.0,6,0,1,0,0.0,0
2,0.214501,32,0,0.211999,3716.0,8,0,0,0,2.0,0
3,1.000000,60,0,118.000000,NaN,5,0,0,0,0.0,0
4,0.230493,60,0,1.017328,3000.0,10,0,1,0,0.0,0


## 1. Разведочный анализ данных

Прежде чем строить модель, надо изучить датасет: размер, типы признаков, наличие пропусков, баланс классов и распределения

In [125]:
print(f"Размер датасета: {df.shape[0]}, {df.shape[1]}")
print()
df.info()

Размер датасета: 120000, 11

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   RevolvingUtilizationOfUnsecuredLines  120000 non-null  float64
 1   age                                   120000 non-null  int64  
 2   NumberOfTime30-59DaysPastDueNotWorse  120000 non-null  int64  
 3   DebtRatio                             120000 non-null  float64
 4   MonthlyIncome                         96325 non-null   float64
 5   NumberOfOpenCreditLinesAndLoans       120000 non-null  int64  
 6   NumberOfTimes90DaysLate               120000 non-null  int64  
 7   NumberRealEstateLoansOrLines          120000 non-null  int64  
 8   NumberOfTime60-89DaysPastDueNotWorse  120000 non-null  int64  
 9   NumberOfDependents                    116872 non-null  float64
 10  SeriousDlqin2yrs                      120000 non-n

In [126]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RevolvingUtilizationOfUnsecuredLines,120000.0,6.128916,253.361490,0.0,0.029593,0.153318,0.557832,50708.0
age,120000.0,52.287842,14.771274,0.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,120000.0,0.420075,4.182138,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,120000.0,352.271245,2093.709509,0.0,0.175330,0.366194,0.860833,329664.0
MonthlyIncome,96325.0,6651.507345,14541.181413,0.0,3400.000000,5390.000000,8238.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,120000.0,8.465692,5.161693,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,120000.0,0.264942,4.158242,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,120000.0,1.018167,1.133055,0.0,0.000000,1.000000,2.000000,54.0
NumberOfTime60-89DaysPastDueNotWorse,120000.0,0.239858,4.144569,0.0,0.000000,0.000000,0.000000,98.0
NumberOfDependents,116872.0,0.757333,1.115337,0.0,0.000000,0.000000,1.000000,20.0


Надо проверить соотношение классов, чтобы корректно выбрать метрики и настройки модели

In [127]:
class_counts = df["SeriousDlqin2yrs"].value_counts()
print(class_counts)
print()
print(f"Доля класса 1 (просрочка)): {class_counts[1] / len(df):.2%}")
print(f"Доля класса 0 (нет просрочки): {class_counts[0] / len(df):.2%}")

SeriousDlqin2yrs
0    111979
1      8021
Name: count, dtype: int64

Доля класса 1 (просрочка)): 6.68%
Доля класса 0 (нет просрочки): 93.32%


### Пропуски в данных

In [128]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Пропуски": missing, "% от всех": missing_pct})
print(missing_df[missing_df["Пропуски"] > 0])

                    Пропуски  % от всех
MonthlyIncome          23675      19.73
NumberOfDependents      3128       2.61


### Выбросы и распределения признаков

Также надо посмотреть на статистики данных. Я обращаю внимание на min/max значения, которые могут указывать на выбросы

In [129]:
for col in df.columns:
    q01 = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    print(f"{col:>45s}  |  min={df[col].min():>12.2f}  q01={q01:>12.2f}  median={df[col].median():>12.2f}  q99={q99:>12.2f}  max={df[col].max():>12.2f}")

         RevolvingUtilizationOfUnsecuredLines  |  min=        0.00  q01=        0.00  median=        0.15  q99=        1.09  max=    50708.00
                                          age  |  min=        0.00  q01=       24.00  median=       52.00  q99=       87.00  max=      109.00
         NumberOfTime30-59DaysPastDueNotWorse  |  min=        0.00  q01=        0.00  median=        0.00  q99=        4.00  max=       98.00
                                    DebtRatio  |  min=        0.00  q01=        0.00  median=        0.37  q99=     4977.00  max=   329664.00
                                MonthlyIncome  |  min=        0.00  q01=        0.00  median=     5390.00  q99=    25000.00  max=  3008750.00
              NumberOfOpenCreditLinesAndLoans  |  min=        0.00  q01=        0.00  median=        8.00  q99=       25.00  max=       58.00
                      NumberOfTimes90DaysLate  |  min=        0.00  q01=        0.00  median=        0.00  q99=        3.00  max=       98.00
      

## 2. Предобработка данных

По результатам анализа данных обнаружены следующие аномальные значения:
- RevolvingUtilizationOfUnsecuredLines - доля, но есть значения, равные 50708. Сделаю клиппинг по p99
- DebtRatio - тоже доля, но максимальное значение = 329664. Сделаю клиппинг по p99
- age — есть значение 0, что невозможно (возраст заёмщика не может быть равен нулю). Заменяю на медиану
- Столбцы просрочек (NumberOfTime30-59DaysPastDueNotWorse, NumberOfTimes90DaysLate, NumberOfTime60-89DaysPastDueNotWorse) — значение 98 выглядит странно. Делаю клиппинг по p99

In [130]:
clip_cols = [
    "RevolvingUtilizationOfUnsecuredLines",
    "DebtRatio",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
]

clip_upper = {}
for col in clip_cols:
    upper = df[col].quantile(0.99)
    clip_upper[col] = upper
    before_clip = (df[col] > upper).sum()
    df[col] = df[col].clip(upper=upper)
    print(f"{col}: клипировано {before_clip} значений (upper={upper})")

df.loc[df["age"] == 0, "age"] = df["age"].median()

RevolvingUtilizationOfUnsecuredLines: клипировано 1200 значений (upper=1.0944386634299974)
DebtRatio: клипировано 1199 значений (upper=4977.0)
NumberOfTime30-59DaysPastDueNotWorse: клипировано 671 значений (upper=4.0)
NumberOfTimes90DaysLate: клипировано 690 значений (upper=3.0)
NumberOfTime60-89DaysPastDueNotWorse: клипировано 610 значений (upper=2.0)


### Заполнение пропусков

- MonthlyIncome (19.7% пропусков) — заполняю медианой
- NumberOfDependents (2.6% пропусков) — заполняем медианой

In [131]:
fill_values = {}
for col in ["MonthlyIncome", "NumberOfDependents"]:
    median_val = df[col].median()
    fill_values[col] = median_val
    n_missing = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: заполнено {n_missing} пропусков медианой ({median_val})")

MonthlyIncome: заполнено 23675 пропусков медианой (5390.0)
NumberOfDependents: заполнено 3128 пропусков медианой (0.0)


Логистическая регрессия чувствительна к масштабу признаков, поэтому применяем StandardScaler

In [132]:
X = df.drop("SeriousDlqin2yrs", axis=1)
y = df["SeriousDlqin2yrs"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y.shape}")
X_scaled.describe().T

X shape: (120000, 10)
y shape: (120000,)


,count,mean,std,min,25%,50%,75%,max
RevolvingUtilizationOfUnsecuredLines,120000.0,-8.715991e-17,1.000004,-0.908836,-0.824736,-0.473120,0.676467,2.201451
age,120000.0,8.763360e-17,1.000004,-2.118303,-0.764248,-0.019517,0.725214,3.839542
NumberOfTime30-59DaysPastDueNotWorse,120000.0,3.931670e-17,1.000004,-0.369220,-0.369220,-0.369220,-0.369220,5.639855
DebtRatio,120000.0,-2.842171e-18,1.000004,-0.348609,-0.348416,-0.348205,-0.347659,5.143565
MonthlyIncome,120000.0,3.315866e-18,1.000004,-0.491088,-0.191954,-0.077669,0.076500,230.283285
NumberOfOpenCreditLinesAndLoans,120000.0,6.821210e-17,1.000004,-1.640107,-0.671428,-0.090221,0.490986,9.596563
NumberOfTimes90DaysLate,120000.0,-1.989520e-17,1.000004,-0.215965,-0.215965,-0.215965,-0.215965,7.258324
NumberRealEstateLoansOrLines,120000.0,-2.842171e-18,1.000004,-0.898607,-0.898607,-0.016033,0.866540,46.760348
NumberOfTime60-89DaysPastDueNotWorse,120000.0,2.321106e-17,1.000004,-0.218485,-0.218485,-0.218485,-0.218485,6.670172
NumberOfDependents,120000.0,2.782959e-17,1.000004,-0.666121,-0.666121,-0.666121,0.236982,17.395928


## 3. Разделение на обучающую и тестовую выборки

Использую стратифицированное разделение (`stratify=y`), чтобы сохранить пропорцию классов в обеих частях. Тестовая выборка = 20%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples, доля класса 1: {y_train.mean():.4f}")
print(f"Test: {X_test.shape[0]} samples, доля класса 1: {y_test.mean():.4f}")

Train: 96000 samples, доля класса 1: 0.0668
Test:  24000 samples, доля класса 1: 0.0668


## 4. Обучение логистической регрессии

По условию задачи, выдать кредит просрочившему в 5 раз дороже, чем отказать платёжеспособному. Учитываю это через параметр `class_weight={0: 1, 1: 5}` (штрафует модель за пропуск просрочек в 5 раз сильнее).

In [134]:
model = LogisticRegression(
    class_weight={0: 1, 1: 5},
    max_iter=1000,
    random_state=42,
)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 5}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :t

## 5. Оценка качества модели

Оцениваем на тестовой выборке. Ключевые метрики:
- Recall класса 1 - какую долю реальных просрочников отловили (важно из-за асимметрии стоимости ошибок)
- Confusion matrix - чтобы увидеть FN (пропущенные просрочники) и FP (ложные отказы)
- ROC-AUC - общее качество ранжирования
- Стоимость ошибок - взвешенная оценка с учётом соотношения 5:1

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Нет просрочки (0)", "Просрочка (1)"]))
print()

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn} FP={fp}")
print(f"FN={fn} TP={tp}")
print()

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print()

cost = fn * 5 + fp * 1
print(f"Стоимость ошибок (FN*5 + FP*1)")
print(f"Обученная модель: {cost}")
cost_dummy = y_test.sum() * 5
print(f"DummyClassifier: {cost_dummy} (все просрочники пропущены)")

                   precision    recall  f1-score   support

Нет просрочки (0)       0.96      0.95      0.95     22396
    Просрочка (1)       0.39      0.47      0.42      1604

         accuracy                           0.92     24000
        macro avg       0.67      0.71      0.69     24000
     weighted avg       0.92      0.92      0.92     24000


TN=21218  FP=1178
FN=858  TP=746

Accuracy:  0.9152
Precision: 0.3877
Recall:    0.4651
F1:        0.4229
ROC-AUC:   0.8543

Стоимость ошибок (FN*5 + FP*1)
Обученная модель: 5468
DummyClassifier: 8020 (все просрочники пропущены)


### Подбор оптимального порога классификации

По умолчанию модель предсказывает класс 1, если P(y=1) > 0.5. Но раз пропуск просрочника в 5 раз дороже ложного отказа, оптимальный порог может быть ниже. Перебираю пороги и выбираем тот, который минимизирует стоимость ошибок

In [136]:
thresholds = np.arange(0.05, 0.95, 0.01)
results = []

for t in thresholds:
    preds_t = (y_proba >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, preds_t).ravel()
    cost_t = fn_t * 5 + fp_t * 1
    f1_t = f1_score(y_test, preds_t)
    recall_t = recall_score(y_test, preds_t)
    precision_t = precision_score(y_test, preds_t, zero_division=0)
    results.append((t, cost_t, f1_t, recall_t, precision_t, tp_t, fp_t, fn_t))

results_df = pd.DataFrame(results, columns=["threshold", "cost", "f1", "recall", "precision", "TP", "FP", "FN"])

best_idx = results_df["cost"].idxmin()
best = results_df.loc[best_idx]
best_threshold = best["threshold"]

print(f"Оптимальный порог: {best_threshold:.2f}")
print(f"Стоимость ошибок: {best['cost']:.0f}")
print(f"F1:        {best['f1']:.4f}")
print(f"Recall:    {best['recall']:.4f}")
print(f"Precision: {best['precision']:.4f}")
print(f"TP={best['TP']:.0f}  FP={best['FP']:.0f}  FN={best['FN']:.0f}")

Оптимальный порог: 0.43
Стоимость ошибок: 5334
F1:        0.4220
Recall:    0.5349
Precision: 0.3485
TP=858  FP=1604  FN=746


## 6. Финальная модель и функция predict

Обучаю финальную модель на всех данных. Функция predict включает полный пайплайн предобработки: обработку выбросов, заполнение пропусков и масштабирование.

In [137]:
final_model = LogisticRegression(
    class_weight={0: 1, 1: 5},
    max_iter=1000,
    random_state=42,
)
final_model.fit(X_scaled, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 5}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :t

In [138]:
def predict(model, data_path):
    df = pd.read_csv(data_path)
    X = df.drop("SeriousDlqin2yrs", axis=1)

    for col, upper in clip_upper.items():
        X[col] = X[col].clip(upper=upper)

    X.loc[X["age"] == 0, "age"] = fill_values.get("age", X["age"].median())

    for col, val in fill_values.items():
        X[col] = X[col].fillna(val)

    X_transformed = pd.DataFrame(scaler.transform(X), columns=X.columns)
    proba = model.predict_proba(X_transformed)[:, 1]
    return (proba >= best_threshold).astype(int)


In [139]:
preds = predict(final_model, train_file)

cm_final = confusion_matrix(y, preds)
tn_f, fp_f, fn_f, tp_f = cm_final.ravel()
cost_final = fn_f * 5 + fp_f * 1
cost_baseline = y.sum() * 5

print(f"Accuracy: {accuracy_score(y, preds):.4f}")
print(f"Recall (класс 1): {recall_score(y, preds):.4f}")
print(f"Precision (класс 1): {precision_score(y, preds):.4f}")
print(f"F1 (класс 1): {f1_score(y, preds):.4f}")
print()
print(f"Стоимость ошибок (FN×5 + FP×1): {cost_final}")
print(f"Baseline (всегда 0): {cost_baseline}")
print(f"Улучшение: {(1 - cost_final / cost_baseline):.1%}")

Accuracy: 0.9015
Recall (класс 1): 0.5471
Precision (класс 1): 0.3490
F1 (класс 1): 0.4261

Стоимость ошибок (FN×5 + FP×1): 26350
Baseline (всегда 0): 40105
Улучшение: 34.3%


## Вывод

Была обучена логистическая регрессия с `class_weight={0: 1, 1: 5}` для предсказания просрочек по кредитам. Основные результаты:

- ROC-AUC = 0.854. Модель хорошо разделяет классы на уровне ранжирования
- Подобран оптимальный порог классификации, минимизирующий бизнес-стоимость ошибок с учётом того, что пропуск просрочника в 5 раз дороже ложного отказа
- По основной метрике — стоимости ошибок (FN * 5 + FP * 1) — модель значительно лучше baseline.
- Accuracy при этом чуть ниже baseline (90% против 93%), но это ожидаемый компромисс: модель сознательно жертвует частью accuracy ради выявления просрочников и снижения убытков банка.